In [1]:
import json
import gzip
import os
from collections import defaultdict
from config import *

In [2]:
def analyze_trace(trace_path):
    
    open_func = gzip.open if trace_path.endswith('.gz') else open
    
    with open_func(trace_path, 'rt') as f:
        trace_data = json.load(f)
        
    events = trace_data if isinstance(trace_data, list) else trace_data.get('traceEvents', [])
    
    # Categorize durations (in microseconds)
    durations = defaultdict(float)
    
    for event in events:
        # We only care about Complete (X) events that have a duration on the GPU
        if event.get('ph') == 'X' and 'dur' in event:
            name = event.get('name', '').lower()
            dur = event.get('dur', 0)
            
            # Categorize NCCL (Communication)
            if 'nccl' in name:
                durations['communication'] += dur
            # Categorize Compute (Matmuls, Linear layers)
            elif 'gemm' in name or 'linear' in name or 'mm' in name:
                durations['computation'] += dur

    comm_time = durations['communication']
    comp_time = durations['computation']
    
    print(f"--- Trace Analysis: {trace_path} ---")
    print(f"Total Communication (NCCL): {comm_time / 1000:.2f} ms")
    print(f"Total Computation (GEMM)  : {comp_time / 1000:.2f} ms")
    
    if comp_time > 0:
        ratio = comm_time / comp_time
        print(f"Comm-to-Comp Ratio      : {ratio:.2f}x")
    else:
        print("No computation found.")

In [3]:
def get_trace_files(trace_dir=trace_dir, base_dir=trace_dir):
    file_paths=[]
    for dirpath, dirnames, filenames in os.walk(trace_dir):
        for filename in filenames:
            if filename.endswith('json.gz') and 'rank' in filename:
                full_path = os.path.join(dirpath, filename)
                rel_path = os.path.relpath(full_path, start=base_dir)
                file_paths.append(rel_path)
    return file_paths

In [4]:
# Analyze all traces
balanced_traces = get_trace_files(trace_dir=trace_path_balanced, base_dir='./')
print('\n... Balanced Model ...')
for t in balanced_traces: analyze_trace(t)
    
imbalanced_traces = get_trace_files(trace_dir=trace_path_imbalanced, base_dir='./')
print('\n... Imbalanced Model ...')
for t in imbalanced_traces: analyze_trace(t)


... Balanced Model ...
--- Trace Analysis: vllm_benchmarking/traces/gpu2_batch16_samples100_imbalance0_layers8_n8_k1_hiddensize2048_intermediatesize8192/dp0_pp0_tp0_dcp0_ep0_rank0.1786365903954268282.pt.trace.json.gz ---
Total Communication (NCCL): 1083.37 ms
Total Computation (GEMM)  : 488.49 ms
Comm-to-Comp Ratio      : 2.22x
--- Trace Analysis: vllm_benchmarking/traces/gpu2_batch16_samples100_imbalance0_layers8_n8_k1_hiddensize2048_intermediatesize8192/dp0_pp0_tp1_dcp0_ep1_rank1.1786365903940158602.pt.trace.json.gz ---
Total Communication (NCCL): 701.51 ms
Total Computation (GEMM)  : 475.08 ms
Comm-to-Comp Ratio      : 1.48x

... Imbalanced Model ...
--- Trace Analysis: vllm_benchmarking/traces/gpu2_batch16_samples100_imbalance100_layers8_n8_k1_hiddensize2048_intermediatesize8192/dp0_pp0_tp1_dcp0_ep1_rank1.1786366089915580517.pt.trace.json.gz ---
Total Communication (NCCL): 59.74 ms
Total Computation (GEMM)  : 464.99 ms
Comm-to-Comp Ratio      : 0.13x
--- Trace Analysis: vllm_bench